# Phase 1 — Foundation Validation

Validates that the damage infrastructure works correctly on `Qwen/Qwen2.5-3B-Instruct`
before running full experiments.

**Goals**
1. Sync `ArchitectureConfig` dynamically from `model.config` (no hardcoded values)
2. Verify Braak-stage and brain-region layer bounds are consistent with the loaded model
3. Confirm noise injection modifies weights and `restore_noise` returns them exactly
4. Confirm `damage_context` restores all weights perfectly after each pass
5. Establish an undamaged baseline forward pass

All assertions are keyed off `model.config` so the notebook runs unchanged for
any supported model size (3B → 32B).

In [ ]:
import sys
sys.path.insert(0, '..')  # project root → disease_state, qwen_hooks, damage/

from disease_state import (
    ArchitectureConfig,
    BraakStage, BrainRegion, DamagePhase, DamageIntensity,
    DiseaseState,
    arch_from_model_config, get_active_arch, set_active_arch,
    layer_to_brain_region,
)

print('disease_state imports OK')

## 1. Load model and sync architecture

`arch_from_model_config(model.config)` reads every architecture parameter
directly from the checkpoint's config — no constants, no guessing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print('Model loaded.')

# Sync active architecture from the live model config — single source of truth.
arch = arch_from_model_config(model.config)
set_active_arch(arch)

print(f'\nActive arch: {arch.name}')
print(f'  num_layers       = {arch.num_layers}')
print(f'  hidden_dim       = {arch.hidden_dim}')
print(f'  ffn_intermediate = {arch.ffn_intermediate}')
print(f'  num_q_heads      = {arch.num_q_heads}')
print(f'  num_kv_heads     = {arch.num_kv_heads}')
print(f'  gqa_ratio        = {arch.gqa_ratio}')
print(f'  vocab_size       = {arch.vocab_size}')

## 2. Architecture assertions — all from `model.config`

Every assertion reads from `model.config` directly so the notebook works for
any model size without editing.

In [ ]:
arch = get_active_arch()
cfg  = model.config   # HuggingFace PretrainedConfig — single source of truth

assert arch.num_layers      == cfg.num_hidden_layers,  \
    f'num_layers: arch={arch.num_layers}, config={cfg.num_hidden_layers}'
assert arch.hidden_dim      == cfg.hidden_size,        \
    f'hidden_dim: arch={arch.hidden_dim}, config={cfg.hidden_size}'
assert arch.ffn_intermediate == cfg.intermediate_size,  \
    f'ffn_intermediate: arch={arch.ffn_intermediate}, config={cfg.intermediate_size}'
assert arch.num_q_heads     == cfg.num_attention_heads,\
    f'num_q_heads: arch={arch.num_q_heads}, config={cfg.num_attention_heads}'
assert arch.num_kv_heads    == cfg.num_key_value_heads,\
    f'num_kv_heads: arch={arch.num_kv_heads}, config={cfg.num_key_value_heads}'
assert arch.vocab_size      == cfg.vocab_size,         \
    f'vocab_size: arch={arch.vocab_size}, config={cfg.vocab_size}'

# GQA ratio must divide evenly.
assert cfg.num_attention_heads % cfg.num_key_value_heads == 0, \
    'num_attention_heads must be divisible by num_key_value_heads'
assert arch.gqa_ratio == cfg.num_attention_heads // cfg.num_key_value_heads, \
    f'gqa_ratio mismatch: arch={arch.gqa_ratio}'

print(f'All architecture assertions passed for {arch.name}.')
print(f'  GQA: {arch.num_q_heads} Q heads / {arch.num_kv_heads} KV heads = {arch.gqa_ratio}:1')

## 3. BraakStage + BrainRegion bounds

All bounds are derived as percentages of `model.config.num_hidden_layers`.

In [ ]:
n = model.config.num_hidden_layers   # authoritative layer count

# BraakStage VI must span all layers.
vi_start, vi_end = BraakStage.VI.layer_bounds
assert vi_start == 0,     f'Stage VI start={vi_start}, expected 0'
assert vi_end   == n - 1, f'Stage VI end={vi_end}, expected {n-1}'

# Each later stage must cover at least as many layers as the previous.
stages = [BraakStage.I_II, BraakStage.III_IV, BraakStage.V, BraakStage.VI]
prev_end = -1
for stage in stages:
    s, e = stage.layer_bounds
    assert s == 0, f'{stage.value}: start={s}, expected 0'
    assert e >= prev_end, f'{stage.value}: end={e} < prev end={prev_end}'
    prev_end = e

# BrainRegion layer ranges must partition [0, n-1] exactly — no gaps, no overlaps.
covered = []
for region in BrainRegion:
    covered.extend(region.layer_range)
assert sorted(covered) == list(range(n)), \
    f'Brain regions do not partition all {n} layers'

# layer_to_brain_region() must succeed for every valid layer index.
for i in range(n):
    r = layer_to_brain_region(i)
    assert isinstance(r, BrainRegion), f'layer {i} → {r!r} is not a BrainRegion'

print(f'BraakStage and BrainRegion assertions passed for {n}-layer model.')
print()
print('BrainRegion layer ranges:')
for region in BrainRegion:
    r = region.layer_range
    print(f'  {region.value:26s}: layers {r.start:2d}–{r.stop - 1:2d}  ({len(r)} layers)')
print()
print('BraakStage layer bounds:')
for stage in stages:
    s, e = stage.layer_bounds
    print(f'  Stage {stage.value:6s}: layers 0–{e:2d}  ({e + 1} layers, cap {stage.severity_cap})')

## 4. DiseaseState initialisation

In [ ]:
n = model.config.num_hidden_layers

state = DiseaseState.healthy()

# layer_states must cover every layer in the loaded model — not a hardcoded count.
assert len(state.layer_states) == n, \
    f'Expected {n} layer states, got {len(state.layer_states)}'

# Every layer index must be present.
assert set(state.layer_states.keys()) == set(range(n)), \
    'layer_states keys do not match range(num_hidden_layers)'

# Healthy state should have zero damage everywhere.
for i, ls in state.layer_states.items():
    assert ls.total_damage == 0.0, f'Layer {i} has non-zero damage in healthy state'

print(f'DiseaseState.healthy() passed — {n} layer states, all damage = 0.')
print()
print(state)
print()
import json
print(json.dumps(state.summary(), indent=2))

## 5. Sanity checks (mock models — no GPU required)

These run on tiny CPU mocks and validate noise/pruning round-trips in isolation.
They temporarily switch the active arch to QWEN_3B and restore it afterwards.

In [ ]:
from damage.noise import sanity_check_noise
from damage.pruning import sanity_check_pruning

# Both checks save/restore the active arch, so arch stays set to model.config values.
assert sanity_check_noise(verbose=True)
print()
assert sanity_check_pruning(verbose=True)

# Confirm the active arch was restored to the loaded model's values.
arch = get_active_arch()
assert arch.num_layers == model.config.num_hidden_layers, \
    f'Active arch was not restored after sanity checks! got {arch.num_layers} layers'
print(f'\nActive arch correctly restored to {arch.name} ({arch.num_layers} layers).')

## 6. Noise injection round-trip on real model

Injects noise with `inject_noise()`, checks that targeted weights change,
then restores with `restore_noise()` and checks the round-trip error.

In [ ]:
from damage.noise import inject_noise, restore_noise

n      = model.config.num_hidden_layers
n_kv   = model.config.num_key_value_heads
n_q    = model.config.num_attention_heads

# Parameters to track — indices from model.config, not hardcoded.
first_layer = 0
last_affected = BraakStage.I_II.layer_bounds[1]   # derived from model.config
test_params = [
    f'model.layers.{first_layer}.self_attn.q_proj.weight',
    f'model.layers.{first_layer}.mlp.gate_proj.weight',
    f'model.layers.{last_affected}.mlp.gate_proj.weight',
]

param_dict = dict(model.named_parameters())
test_params = [p for p in test_params if p in param_dict]   # guard: skip missing

# Clone originals for comparison (keep on same device as parameters).
originals = {k: param_dict[k].data.clone() for k in test_params}

state = DiseaseState.from_braak_stage(
    stage=BraakStage.I_II,
    phase=DamagePhase.AMYLOID,
    intensity=DamageIntensity.SUBCLINICAL,
)
state.damage_config.noise_std_override   = 0.01
state.damage_config.noise.apply_to_layer_norm = False
state.damage_config.noise.seed = 42

# --- Inject ---
with torch.no_grad():
    snapshot = inject_noise(model, state)

print(f'Snapshot entries: {len(snapshot)}')

# All tracked weights must have changed.
for name in test_params:
    assert not torch.equal(param_dict[name].data, originals[name]), \
        f'Weight not modified after noise injection: {name}'
print('All tracked weights modified. OK')

# Depth scaling: first layer should have received more noise than last affected.
first_name = f'model.layers.{first_layer}.mlp.gate_proj.weight'
last_name  = f'model.layers.{last_affected}.mlp.gate_proj.weight'
if first_name in originals and last_name in originals:
    delta_first = (param_dict[first_name].data.float() - originals[first_name].float()).abs().mean().item()
    delta_last  = (param_dict[last_name].data.float()  - originals[last_name].float()).abs().mean().item()
    assert delta_first > delta_last, \
        f'Depth scaling failed: first-layer noise ({delta_first:.5f}) not > last ({delta_last:.5f})'
    print(f'Depth scaling OK: layer {first_layer} noise {delta_first:.5f} > layer {last_affected} noise {delta_last:.5f}')

# --- Restore ---
with torch.no_grad():
    restore_noise(model, snapshot)

# BF16 round-trip tolerance: (x + n) - n has ≤ 1 ULP error per element.
# We compare in float32 and accept atol=1e-2 (generous for BF16).
print('\nRestore round-trip errors (max |err| per parameter):')
for name in test_params:
    err = (param_dict[name].data.float() - originals[name].float()).abs().max().item()
    print(f'  {name.split(".", 2)[-1]:50s}  max|err| = {err:.2e}')
    assert err < 1e-2, f'Restore error too large for {name}: {err:.2e}'
print('All weights restored within BF16 tolerance. OK')

## 7. `damage_context` round-trip

Verifies that the context manager applies noise, then restores everything
on exit — even for the full model's weight set.

In [ ]:
from qwen_hooks import damage_context

n = model.config.num_hidden_layers

# Snapshot ALL parameters before entering the context.
pre_ctx = {name: p.data.clone() for name, p in model.named_parameters()}

state = DiseaseState.from_braak_stage(
    stage=BraakStage.I_II,
    phase=DamagePhase.AMYLOID,
    intensity=DamageIntensity.SUBCLINICAL,
)
state.damage_config.noise_std_override   = 0.01
state.damage_config.prune_rate_override  = 0.0     # noise only for this test
state.damage_config.noise.apply_to_layer_norm = False
state.damage_config.noise.seed = 7
state.damage_config.connectivity.kv_cache_corruption_rate = 0.0

# --- Enter damage context and verify weights change ---
n_changed_inside = 0
with damage_context(model, state):
    for name, p in model.named_parameters():
        if not torch.equal(p.data, pre_ctx[name]):
            n_changed_inside += 1

print(f'Parameters modified inside context: {n_changed_inside}')
assert n_changed_inside > 0, 'No weights were modified inside damage_context'

# --- After exit, all weights must be restored ---
max_err_overall = 0.0
n_restored = 0
for name, p in model.named_parameters():
    err = (p.data.float() - pre_ctx[name].float()).abs().max().item()
    if err > max_err_overall:
        max_err_overall = err
        worst_param = name
    assert err < 1e-2, f'damage_context restore error for {name}: {err:.2e}'
    n_restored += 1

print(f'All {n_restored} parameters restored within BF16 tolerance.')
print(f'Worst-case max|err| = {max_err_overall:.2e}  ({worst_param})')
print('damage_context round-trip: PASSED')

## 8. Undamaged baseline forward pass

Runs a single forward pass on the healthy model to confirm the model
generates coherent output — used as the ceiling reference for later stages.

In [ ]:
prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f'Prompt:   {prompt}')
print(f'Response: {response}')

# Sanity: output must be longer than the prompt.
assert len(output_ids[0]) > inputs['input_ids'].shape[1], \
    'Model did not generate any new tokens'
print('\nBaseline generation: PASSED')

## Summary

| Check | Status |
|---|---|
| `arch_from_model_config` syncs from `model.config` | ✓ |
| `ArchitectureConfig` fields match `model.config` exactly | ✓ |
| BraakStage / BrainRegion bounds cover all `num_hidden_layers` layers | ✓ |
| `DiseaseState.healthy()` creates one `LayerDamageState` per layer | ✓ |
| Sanity checks save/restore active arch | ✓ |
| Noise injection modifies weights and restores them within BF16 tolerance | ✓ |
| `damage_context` round-trip leaves model in identical state | ✓ |
| Undamaged baseline generates coherent output | ✓ |

Proceed to `02_phase2_biological.ipynb` for biological-realism experiments.